# Introdução ao PyTorch — passo a passo para iniciantes

## O que é o PyTorch?
PyTorch é uma biblioteca Python para **machine learning** e **deep learning**. Ela trabalha com **tensores** (arrays numéricos multidimensionais) e calcula **gradientes automaticamente** (autograd), o que permite treinar modelos ajustando parâmetros com base no erro.

## Objetivo deste notebook
Ao final, você vai entender a sequência básica de qualquer treino em PyTorch:
1. **Configurar** o ambiente (imports, semente, CPU/GPU)
2. **Manipular tensores** (criar, somar, multiplicar)
3. **Definir inputs** do problema
4. **Treinar** com forward → loss → backward → update
5. **Visualizar** como loss e peso evoluem
6. **Carregar** o dataset MNIST com `torchvision`
7. **Treinar um MLP** e um modelo customizado (`GatedSkipNet`)

## Como usar
Execute as células **na ordem**, de cima para baixo.  
**Kernel:** selecione **Python (classes exercises)** no canto superior direito.

---

### Roteiro

| Passo | Tópico | O que você vai ver |
|-------|--------|--------------------|
| 1 | Setup | `Using device: cpu` (ou `cuda`) |
| 2 | Tensores | Operações com `tensor([...])` |
| 3 | Inputs | Valores do experimento impressos |
| 4 | Treino | `Step 0 → weight = 1.85, loss = 20.25` |
| 5 | Gráficos | Curvas de loss e weight |
| 6 | MNIST | Imagens 28×28 + índices 6000 |
| 7 | DataLoaders | `images.shape: (128, 1, 28, 28)` |
| 8 | MLP | `Sequential(Flatten, Linear, ReLU, …)` |
| 9 | Forward pass | `logits.shape: (128, 10)` |
| 10 | Loss + Adam | `loss value for this batch: 2.35` |
| 11 | Treino completo | `train loss / val loss / val acc` por época |
| 12 | GatedSkipNet | Modelo com gate, skip e `torch.cat` |
| 13 | Recap | 12 perguntas de revisão com respostas |


---
## Passo 1 — Configurar o ambiente

### O que é?
A primeira célula de código **importa bibliotecas**, **fixa a semente aleatória** e **escolhe onde rodar** (CPU ou GPU).

### Por que usamos?

| Elemento | Motivo |
|----------|--------|
| `import torch` | Biblioteca principal para tensores e redes neurais |
| `set_seed(33)` | Garante que o mesmo código produza os mesmos números (reprodutibilidade) |
| `device` | GPU acelera treinos grandes; CPU funciona para exemplos pequenos |
| `KMP_DUPLICATE_LIB_OK` | Evita erro de bibliotecas duplicadas no Windows (não altera o algoritmo) |

### Output esperado

```text
Python em uso: ...\classes exercises\.venv\Scripts\python.exe
Using device: cpu
```

Se aparecer outro caminho de Python, troque o kernel para **Python (classes exercises)**.

### Objetivo
Deixar o ambiente pronto e confirmar que PyTorch está funcionando no interpretador correto.

In [36]:
import sys
import subprocess
import os
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib>=3.8"])
    import matplotlib.pyplot as plt

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(f"Python em uso: {sys.executable}")


def set_seed(seed=33):
    """Fixa sementes para reproduzir os mesmos resultados em cada execução."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(33)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Python em uso: c:\Users\CALIXTOI\Downloads\Github\Personal Github\statistics-learning-2\classes exercises\.venv\Scripts\python.exe
Using device: cpu


---
## Passo 2 — Tensores: a estrutura básica do PyTorch

### O que é?
Um **tensor** é como um array NumPy, mas o PyTorch sabe calcular **gradientes** sobre ele quando `requires_grad=True`. É a unidade fundamental de dados e parâmetros.

### Por que usamos?
- Toda entrada, saída e peso de rede neural é um tensor.
- Operações (`+`, `*`, matriz identidade, zeros) funcionam de forma vetorizada (rápida).

### Output esperado

```text
x: tensor([1., 2., 3.])
x + y = tensor([11., 22., 33.])   # soma elemento a elemento
x * y = tensor([10., 40., 90.])   # produto elemento a elemento
tensor([[1., 0., ...]])           # matriz identidade 10x10
tensor([0., 0., 0.])              # vetor de zeros com mesmo tamanho de x
```

### Objetivo
Familiarizar-se com a criação e operações simples antes de treinar um modelo.

In [37]:
# Criar tensores a partir de listas Python
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])

print("x:", x)
print("y:", y)
print("x + y =", x + y)   # soma elemento a elemento
print("x * y =", x * y)   # produto elemento a elemento

# Matriz identidade 10x10 (1 na diagonal, 0 no resto)
z = torch.eye(10)
print(z)

# Vetor de zeros com o mesmo formato de x
z = torch.zeros_like(x)
print(z)

x: tensor([1., 2., 3.])
y: tensor([10., 20., 30.])
x + y = tensor([11., 22., 33.])
x * y = tensor([10., 40., 90.])
tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])
tensor([0., 0., 0.])


---
## Passo 3 — Definir os inputs do experimento

### O que é?
Vamos treinar o modelo mais simples possível:

- **Modelo:** `prediction = weight × input`
- **Perda (MSE):** `loss = (prediction − target)²`

O PyTorch calcula automaticamente **quanto mudar o peso** para reduzir a perda (gradiente).

### Por que usamos?
Separar os **inputs** do **código de treino** facilita testar valores diferentes sem mexer no loop.

Este mini-exemplo replica o ciclo de **todo** treino de rede neural:

| Etapa | O que faz |
|-------|-----------|
| Forward | Calcula predição e loss |
| Backward | Calcula gradientes com `loss.backward()` |
| Update | Ajusta o peso na direção que reduz o erro |

### Inputs (edite na célula seguinte)

| Variável | Valor | Papel |
|----------|-------|-------|
| `INPUT_VALUE` | `3.0` | entrada fixa |
| `TARGET_VALUE` | `6.0` | valor que queremos atingir |
| `INITIAL_WEIGHT` | `0.5` | peso antes do treino |
| `LEARNING_RATE` | `0.05` | tamanho de cada passo |
| `NUM_STEPS` | `10` | quantas vezes repetir o ciclo |

**Verificação no step 0:** com esses valores, após o primeiro update → `weight = 1.85`, `loss = 20.25`.

- `prediction = 0.5 × 3 = 1.5`
- `loss = (1.5 − 6)² = 20.25`
- peso ótimo final = `target / input = 6 / 3 = 2.0`

### Output esperado

```text
Inputs configurados:
  INPUT_VALUE    = 3.0
  ...
  peso ótimo esperado = 2.0
```

### Objetivo
Definir os parâmetros do experimento em um só lugar.

In [38]:
# ========== INPUTS (edite aqui) ==========
INPUT_VALUE = 3.0
TARGET_VALUE = 6.0
INITIAL_WEIGHT = 0.5
LEARNING_RATE = 0.05
NUM_STEPS = 10

# Converte números Python em tensores PyTorch
input_value = torch.tensor(INPUT_VALUE)
target_value = torch.tensor(TARGET_VALUE)

# requires_grad=True → PyTorch rastreia este tensor para calcular gradientes
weight = torch.tensor(INITIAL_WEIGHT, requires_grad=True)

print("Inputs configurados:")
print(f"  INPUT_VALUE    = {INPUT_VALUE}")
print(f"  TARGET_VALUE   = {TARGET_VALUE}")
print(f"  INITIAL_WEIGHT = {INITIAL_WEIGHT}")
print(f"  LEARNING_RATE  = {LEARNING_RATE}")
print(f"  NUM_STEPS      = {NUM_STEPS}")
print(f"  peso ótimo esperado = {TARGET_VALUE / INPUT_VALUE}")

Inputs configurados:
  INPUT_VALUE    = 3.0
  TARGET_VALUE   = 6.0
  INITIAL_WEIGHT = 0.5
  LEARNING_RATE  = 0.05
  NUM_STEPS      = 10
  peso ótimo esperado = 2.0


---
## Passo 4 — Laço de treino

### O que é?
Um **loop** que repete o ciclo forward → backward → update `NUM_STEPS` vezes.

### Por que cada linha importa?

| Código | Função |
|--------|--------|
| `weight.grad.zero_()` | Limpa gradientes do passo anterior |
| `prediction = weight * input_value` | Forward: calcula a saída do modelo |
| `loss = (prediction - target_value) ** 2` | Mede o erro (quadrado da diferença) |
| `loss.backward()` | Backward: calcula ∂loss/∂weight |
| `weight -= LEARNING_RATE * weight.grad` | Update: ajusta o peso |

### Output esperado

```text
Step  0 | weight = 1.85, loss = 20.25
Step  1 | weight = 1.99, loss = 0.20
Step  2 | weight = 2.00, loss = 0.00
...
```

A loss cai rapidamente e o weight converge para **2.0** (valor ótimo).

### Nota importante
A célula de treino **reinicia** `weight = INITIAL_WEIGHT` no início. Assim você pode reexecutá-la sem precisar rodar a célula de inputs de novo.

### Objetivo
Ver na prática como o modelo aprende ajustando um parâmetro para minimizar o erro.

### Por que zerar os gradientes? (`weight.grad.zero_()`)

#### O que é?
Depois de `loss.backward()`, o PyTorch guarda o gradiente em `weight.grad` — ou seja, **quanto o peso deve mudar** para reduzir a loss daquele passo.

`zero_()` **apaga** esse valor antes do próximo passo, colocando o gradiente de volta em zero.

#### Por que usamos?
No PyTorch, `backward()` **acumula** gradientes por padrão (soma o novo gradiente ao que já estava em `.grad`).

Isso é útil em alguns casos (ex.: processar um batch grande em pedaços menores), mas no nosso loop queremos **um gradiente por passo**.

| Situação | O que acontece |
|----------|----------------|
| **Com** `zero_()` | Cada step usa só o gradiente da loss atual → update correto |
| **Sem** `zero_()` | Gradientes de passos anteriores se somam → passos ficam grandes demais ou errados |

#### Exemplo intuitivo (step 1)
- No step 0, suponha `weight.grad = -27`
- Se **não** zerarmos, no step 1 o PyTorch soma o novo gradiente ao `-27` antigo
- O update fica baseado num gradiente **inflado**, não só no erro atual

#### Por que `if weight.grad is not None`?
No **primeiro** passo, ainda não existe gradiente (`weight.grad` é `None`). A partir do step 1, já existe — aí sim zeramos.

#### Objetivo
Garantir que cada iteração do treino responda **apenas à loss daquele momento**, sem “arrastar” gradientes antigos.

In [40]:
# Reinicia o peso — necessário se você reexecutar esta célula sem rodar a de inputs
weight = torch.tensor(INITIAL_WEIGHT, requires_grad=True)

weights_hist, losses_hist = [], []

for step in range(NUM_STEPS):
    # Zera gradientes acumulados — ver seção "Por que zerar os gradientes?"
    if weight.grad is not None:
        weight.grad.zero_()

    prediction = weight * input_value
    loss = (prediction - target_value) ** 2

    loss.backward()

    with torch.no_grad():
        weight -= LEARNING_RATE * weight.grad

    weights_hist.append(weight.item())
    losses_hist.append(loss.item())
    print(f"Step {step:2d} | weight = {weight.item():.2f}, loss = {loss.item():.2f}")

Step  0 | weight = 1.85, loss = 20.25
Step  1 | weight = 1.99, loss = 0.20
Step  2 | weight = 2.00, loss = 0.00
Step  3 | weight = 2.00, loss = 0.00
Step  4 | weight = 2.00, loss = 0.00
Step  5 | weight = 2.00, loss = 0.00
Step  6 | weight = 2.00, loss = 0.00
Step  7 | weight = 2.00, loss = 0.00
Step  8 | weight = 2.00, loss = 0.00
Step  9 | weight = 2.00, loss = 0.00


---
## Passo 5 — Visualizar a convergência

### O que é?
Dois gráficos: **loss por step** e **weight por step**.

### Por que usamos?
Gráficos ajudam a confirmar visualmente que:
- a **loss diminui** (modelo está melhorando);
- o **weight se aproxima de 2.0** (linha tracejada = peso ótimo).

### Output esperado
Um figure com dois painéis:
- **Esquerda:** curva de loss caindo de ~20 para ~0
- **Direita:** curva de weight subindo de 0.5 até 2.0

### Objetivo
Interpretar o comportamento do treino — habilidade essencial em projetos reais de ML.

In [ ]:
optimal_weight = TARGET_VALUE / INPUT_VALUE

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(losses_hist, marker="o", color="#2C3E6B")
axes[0].set_title("Loss por step")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(weights_hist, marker="o", color="#8B0000", label="weight aprendido")
axes[1].axhline(optimal_weight, ls="--", color="gray", label=f"peso ótimo (= {optimal_weight})")
axes[1].set_title("Weight por step")
axes[1].set_xlabel("step")
axes[1].set_ylabel("weight")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Passo 6 — Carregar o dataset MNIST

### O que é?
**MNIST** é um dataset clássico de visão computacional: **70.000 imagens** de dígitos manuscritos (0–9), cada uma com **28×28 pixels** em escala de cinza.

- **Treino:** 60.000 imagens
- **Teste:** 10.000 imagens

Com `torchvision.datasets.MNIST`, o PyTorch **baixa** os arquivos automaticamente (na primeira execução) e devolve pares `(imagem, rótulo)`.

### Por que usamos?

| Componente | Função |
|------------|--------|
| `transforms.ToTensor()` | Converte imagem PIL → tensor `(1, 28, 28)` com valores entre 0 e 1 |
| `transforms.Normalize(mean, std)` | Centraliza os pixels (média ≈ 0, desvio ≈ 1) → treino mais estável |
| `DataLoader` | Agrupa amostras em **batches** para o loop de treino |

**Valores de normalização do MNIST:** `mean=0.1307`, `std=0.3081` (média e desvio do dataset completo).

### Output esperado

```text
MNIST treino (completo): 60000 amostras
MNIST teste:              10000 amostras
6000
tensor([34710, 28543,  ..., 30293])   # 5000 treino + 1000 validação
Train subset: 5000 amostras
Val subset:   1000 amostras
```

Os índices acima aparecem com `set_seed(33)` já executado no Passo 1.

**Nota (Windows):** o `download=True` do torchvision pode falhar por erro de SSL. A célula seguinte baixa do mirror Google (CVDF) ou usa os arquivos já em `./data/MNIST/raw/`.

Também verá um grid com **16 dígitos** de exemplo e o shape de um batch.

### Objetivo
Aprender o pipeline básico de dados em PyTorch: **transform → Dataset → DataLoader → batch**, preparando o terreno para treinar uma rede neural nas próximas aulas.

In [41]:
import gzip
import shutil
import ssl
import urllib.request
from pathlib import Path

# ========== INPUTS MNIST (edite aqui) ==========
DATA_DIR = "./data"
BATCH_SIZE = 64
VALIDATION_SIZE = 1000
TRAINING_SIZE = 5000

MNIST_MIRROR = "https://storage.googleapis.com/cvdf-datasets/mnist/"
MNIST_FILES = [
    "train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte.gz",
]


def ensure_mnist_data(data_dir: str) -> Path:
    """Baixa/extrai MNIST se necessário (contorna erro SSL do torchvision no Windows)."""
    raw_dir = Path(data_dir) / "MNIST" / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    ssl_ctx = ssl._create_unverified_context()

    for filename in MNIST_FILES:
        gz_path = raw_dir / filename
        raw_path = raw_dir / filename.replace(".gz", "")
        if raw_path.exists():
            continue
        if not gz_path.exists():
            url = MNIST_MIRROR + filename
            print(f"Baixando {filename}...")
            with urllib.request.urlopen(url, context=ssl_ctx) as response:
                gz_path.write_bytes(response.read())
        print(f"Extraindo {filename}...")
        with gzip.open(gz_path, "rb") as f_in, open(raw_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

    return raw_dir


mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


def prepare_mnist_splits(
    data_dir=DATA_DIR,
    training_size=TRAINING_SIZE,
    validation_size=VALIDATION_SIZE,
):
    """Carrega MNIST e devolve train_ds, val_ds, test_ds."""
    ensure_mnist_data(data_dir)

    train_full = datasets.MNIST(
        root=data_dir, train=True, download=False, transform=mnist_transform
    )
    test_dataset = datasets.MNIST(
        root=data_dir, train=False, download=False, transform=mnist_transform
    )

    print(f"MNIST treino (completo): {len(train_full)} amostras")
    print(f"MNIST teste:              {len(test_dataset)} amostras")

    indices = torch.randperm(len(train_full), generator=torch.Generator())[
        : training_size + validation_size
    ]
    print(len(indices))
    print(indices)

    small_ds = Subset(train_full, indices)
    train_dataset, validation_dataset = random_split(
        small_ds,
        [training_size, validation_size],
        generator=torch.Generator().manual_seed(33),
    )

    print(f"Train subset: {len(train_dataset)} amostras")
    print(f"Val subset:   {len(validation_dataset)} amostras")

    return train_dataset, validation_dataset, test_dataset


train_ds, val_ds, test_ds = prepare_mnist_splits()

# Inspecionar uma amostra do subset de treino
img, label = train_ds[0]
print(f"\nUma amostra do train_ds:")
print(f"  Imagem shape: {img.shape}")
print(f"  Label: {label}")

# Visualizar 16 exemplos do subset de treino
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    image, digit = train_ds[i]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(str(digit), fontsize=10)
    ax.axis("off")
plt.suptitle("Amostras do MNIST (subset de treino)", fontsize=12)
plt.tight_layout()
plt.show()

RuntimeError: Error downloading train-images-idx3-ubyte.gz:
Tried https://ossci-datasets.s3.amazonaws.com/mnist/, got:
<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Missing Authority Key Identifier (_ssl.c:1028)>
Tried http://yann.lecun.com/exdb/mnist/, got:
HTTP Error 404: Not Found


---
## Passo 7 — DataLoaders (batches para treino)

### O que é?
O `DataLoader` pega um `Dataset` e entrega **batches** `(images, labels)` prontos para o loop de treino.

### Por que usamos?
| Loader | batch_size | shuffle | Motivo |
|--------|------------|---------|--------|
| `train_loader` | 128 | `True` | Embaralha a cada época → modelo não decora ordem |
| `validation_loader` | 256 | `False` | Avaliação estável e reproduzível |
| `test_loader` | 256 | `False` | Idem para o conjunto de teste |

### Output esperado
```text
images.shape: torch.Size([128, 1, 28, 28])
labels.shape: torch.Size([128])
primeiros labels: [3, 7, 1, ...]
```

### Nota
Se você **não rodou o Passo 6**, não tem problema: esta célula chama `prepare_mnist_splits()` automaticamente e cria `train_ds`, `val_ds` e `test_ds`.

### Objetivo
Pegar o primeiro batch com `next(iter(train_loader))` e confirmar o formato dos tensores.

In [ ]:
# Carrega MNIST se ainda não rodou o Passo 6 (define prepare_mnist_splits + datasets)
if "prepare_mnist_splits" not in globals():
    import gzip
    import shutil
    import ssl
    import urllib.request
    from pathlib import Path

    DATA_DIR = "./data"
    VALIDATION_SIZE = 1000
    TRAINING_SIZE = 5000
    MNIST_MIRROR = "https://storage.googleapis.com/cvdf-datasets/mnist/"
    MNIST_FILES = [
        "train-images-idx3-ubyte.gz",
        "train-labels-idx1-ubyte.gz",
        "t10k-images-idx3-ubyte.gz",
        "t10k-labels-idx1-ubyte.gz",
    ]

    def ensure_mnist_data(data_dir: str) -> Path:
        raw_dir = Path(data_dir) / "MNIST" / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)
        ssl_ctx = ssl._create_unverified_context()
        for filename in MNIST_FILES:
            gz_path = raw_dir / filename
            raw_path = raw_dir / filename.replace(".gz", "")
            if raw_path.exists():
                continue
            if not gz_path.exists():
                print(f"Baixando {filename}...")
                with urllib.request.urlopen(MNIST_MIRROR + filename, context=ssl_ctx) as response:
                    gz_path.write_bytes(response.read())
            print(f"Extraindo {filename}...")
            with gzip.open(gz_path, "rb") as f_in, open(raw_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
        return raw_dir

    mnist_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    def prepare_mnist_splits(
        data_dir=DATA_DIR,
        training_size=TRAINING_SIZE,
        validation_size=VALIDATION_SIZE,
    ):
        ensure_mnist_data(data_dir)
        train_full = datasets.MNIST(root=data_dir, train=True, download=False, transform=mnist_transform)
        test_dataset = datasets.MNIST(root=data_dir, train=False, download=False, transform=mnist_transform)
        indices = torch.randperm(len(train_full), generator=torch.Generator())[: training_size + validation_size]
        small_ds = Subset(train_full, indices)
        train_dataset, validation_dataset = random_split(
            small_ds,
            [training_size, validation_size],
            generator=torch.Generator().manual_seed(33),
        )
        return train_dataset, validation_dataset, test_dataset

train_ds, val_ds, test_ds = prepare_mnist_splits()
print(f"Pronto: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

batch_size = 128

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
validation_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

# Primeiro batch do treino
images, labels = next(iter(train_loader))

print(f"images.shape: {images.shape}")   # (128, 1, 28, 28)
print(f"labels.shape: {labels.shape}")   # (128,)
print(f"primeiros labels: {labels[:8].tolist()}")

In [ ]:
def show_images(images, labels, number=8):
    plt.figure(figsize=(12, 3))
    for i in range(number):
        plt.subplot(1, number, i + 1)
        image = images[i].squeeze()
        plt.imshow(image, cmap="gray")
        plt.title(f"Label: {labels[i].item()}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


show_images(images, labels)

---
## Passo 8 — Definir a rede neural (MLP para MNIST)

### O que é?
Um **MLP** achata a imagem 28×28 em um vetor de 784 pixels e passa por camadas lineares.

```text
(batch, 1, 28, 28) → Flatten → Linear(784, 64) → ReLU → Linear(64, 10)
```

A saída tem **10 logits** (um score por dígito 0–9).

### Por que usamos?
| Camada | Função |
|--------|--------|
| `nn.Flatten()` | Converte `(1, 28, 28)` em vetor de 784 |
| `nn.Linear(784, 64)` | Camada oculta com 64 neurônios |
| `nn.ReLU()` | Ativação não linear |
| `nn.Dropout(p)` | Regularização (opcional, só se `dropout > 0`) |
| `nn.Linear(64, 10)` | Saída: 10 classes do MNIST |

### Output esperado
```text
Sequential(
  (0): Flatten()
  (1): Linear(in_features=784, out_features=64, bias=True)
  ...
  (3): Linear(in_features=64, out_features=10, bias=True)
)
```

### Objetivo
Criar `model` com `create_model()` e enviar para `device` (CPU/GPU).


In [ ]:
HIDDEN_SIZE = 64
NUM_CLASSES = 10


def create_model(dropout=0.0):
    """MLP para MNIST: Flatten -> Linear(784, 64) -> ReLU -> [Dropout] -> Linear(64, 10)."""
    layers = [
        nn.Flatten(),
        nn.Linear(28 * 28, HIDDEN_SIZE),
        nn.ReLU(),
    ]
    if dropout > 0:
        layers.append(nn.Dropout(p=dropout))
    layers.append(nn.Linear(HIDDEN_SIZE, NUM_CLASSES))
    return nn.Sequential(*layers)


model = create_model(dropout=0.0).to(device)
print(model)


---
## Passo 9 — Forward pass (uma predição em batch)

### O que é?
O **forward pass** passa um batch de imagens pela rede e produz **logits**.

### Output esperado
```text
logits.shape: torch.Size([128, 10])
classe prevista (argmax): 5
label real: 5
```


In [ ]:
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

model.eval()
with torch.no_grad():
    logits = model(images)

print(f"logits.shape: {logits.shape}")
print(logits[0])


---
## Passo 10 — Loss e otimizador

### O que é?
Para **classificação** (10 classes no MNIST), usamos `CrossEntropyLoss`: compara logits `(batch, 10)` com os rótulos inteiros `(batch,)`.

O **otimizador** (`Adam`) guarda os pesos do modelo e fará os updates depois do `backward()`.

### Por que usamos?
| Objeto | Função |
|--------|--------|
| `nn.CrossEntropyLoss()` | Mede o erro de classificação (já inclui softmax internamente) |
| `torch.optim.Adam(model.parameters(), lr=...)` | Algoritmo de update dos pesos |
| `loss.item()` | Converte o tensor de loss em número Python |

### Atenção
- Use `model.parameters()` (**plural**), não `parameter()`.
- `labels` deve ser `torch.long` (tipo padrão do MNIST).

### Output esperado
```text
loss value for this batch: 2.35
```
(valor alto no início é normal — pesos ainda aleatórios.)


In [ ]:
LEARNING_RATE = 0.001

# Garante modelo e DataLoader (rodar Passos 7–8 antes)
if "model" not in globals():
    model = create_model(dropout=0.0).to(device)
if "train_loader" not in globals():
    if "prepare_mnist_splits" not in globals():
        raise NameError("Execute o Passo 6 ou 7 primeiro.")
    train_ds, val_ds, test_ds = prepare_mnist_splits()
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

model.train()
logits = model(images)
loss = criterion(logits, labels)

print(f"logits.shape: {logits.shape}")
print(f"loss value for this batch: {loss.item():.4f}")


---
## Passo 11 — Avaliar e treinar o modelo

| Função | Papel |
|--------|-------|
| `evaluate()` | Mede loss e acurácia na validação (sem gradientes) |
| `train_model()` | Loop de épocas com early stopping e melhor checkpoint |


In [ ]:
def evaluate(model, data_loader, criterion):
    """Calcula loss media e acuracia em um DataLoader (sem atualizar pesos)."""
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)
            batch_size = images.shape[0]

            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples


In [ ]:
def train_model(
    model,
    name,
    train_loader=train_loader,
    validation_loader=validation_loader,
    weight_decay=0.0,
    label_smoothing=0.0,
    max_epochs=8,
    patience=10,
    learning_rate=0.001,
):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = {
        "train_loss": [],
        "validation_loss": [],
        "validation_accuracy": [],
    }

    best_validation_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        total_train_loss = 0.0
        total_train_examples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size = images.shape[0]
            total_train_loss += loss.item() * batch_size
            total_train_examples += batch_size

        average_train_loss = total_train_loss / total_train_examples
        validation_loss, validation_accuracy = evaluate(
            model, validation_loader, criterion
        )

        history["train_loss"].append(average_train_loss)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)

        print(
            f"{name:5s} epoch {epoch + 1:2d}/{max_epochs} | "
            f"train loss = {average_train_loss:.4f} | "
            f"val loss = {validation_loss:.4f} | "
            f"val acc = {validation_accuracy:.4f}"
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_model_state = {
                key: value.cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping.")
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, history


model_baseline = create_model(dropout=0.0)
model_baseline, history_baseline = train_model(model_baseline, name="MLP")


---
## Passo 12 — `GatedSkipNet`

```
h = stem(x)
mixed = gate * branch_a(h) + (1-gate) * branch_b(h)
combined = dropout(mixed + skip(h))
features = concat(bottleneck(combined), h)  →  output
```


In [ ]:
class GatedSkipNet(nn.Module):
    """Duas ramificacoes misturadas por gate + skip connection + bottleneck."""

    def __init__(
        self,
        hidden_size=64,
        bottleneck_size=32,
        dropout=0.2,
        num_classes=10,
    ):
        super().__init__()
        self.flatten = nn.Flatten()

        self.stem = nn.Linear(28 * 28, hidden_size)
        self.branch_a = nn.Linear(hidden_size, hidden_size)
        self.branch_b = nn.Linear(hidden_size, hidden_size)
        self.gate = nn.Linear(hidden_size, hidden_size)
        self.skip = nn.Linear(hidden_size, hidden_size)

        self.dropout = nn.Dropout(p=dropout)
        self.bottleneck = nn.Linear(hidden_size, bottleneck_size)
        self.output = nn.Linear(bottleneck_size + hidden_size, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        h = torch.relu(self.stem(x))

        a = torch.relu(self.branch_a(h))
        b = torch.tanh(self.branch_b(h))
        gate = torch.sigmoid(self.gate(h))

        mixed = gate * a + (1 - gate) * b
        skip = torch.relu(self.skip(h))
        combined = self.dropout(mixed + skip)

        bottleneck = torch.relu(self.bottleneck(combined))
        features = torch.cat([bottleneck, h], dim=1)

        return self.output(features)


In [ ]:
complex_model = GatedSkipNet().to(device)
print(complex_model)

images, labels = next(iter(train_loader))
with torch.no_grad():
    logits = complex_model(images.to(device))
print(f"logits.shape: {logits.shape}")

complex_model, history_complex = train_model(complex_model, name="Gated")



---
## Recap — perguntas e respostas

Revise estes pontos antes da prova ou do próximo notebook.

---

### 1. What is the usual order in one training step?

**Ordem usual:**

1. **Forward pass** — `logits = model(images)`
2. **Calcular a loss** — `loss = criterion(logits, labels)`
3. **Zerar gradientes antigos** — `optimizer.zero_grad()`
4. **Calcular gradientes** — `loss.backward()`
5. **Atualizar pesos** — `optimizer.step()`

Resumo: **forward → loss → zero_grad → backward → step**.

---

### 2. Why is `optimizer.zero_grad()` needed before calculating new gradients?

No PyTorch os gradientes **se acumulam**: cada `loss.backward()` **soma** ao que já está em `param.grad`.

Sem `zero_grad()`, o passo seguinte mistura o gradiente do batch atual com o do anterior → `optimizer.step()` usa um gradiente **errado**.

`zero_grad()` garante que só o backward **deste** batch entra na atualização.

---

### 3. Which line actually updates the model parameters?

```python
optimizer.step()
```

Ela aplica a regra do otimizador (ex.: Adam) usando os valores em `param.grad`.

---

### 4. Which line calculates gradients?

```python
loss.backward()
```

O autograd percorre o grafo da loss e preenche `.grad` de cada parâmetro usado no forward.

---

### 5. In a 10-class digit classification problem, what should the output shape be for a batch of 128 images?

```text
(128, 10)
```

- **128** = tamanho do batch
- **10** = um score por classe (dígitos 0–9)

---

### 6. What do the 10 output values represent before applying a probability interpretation?

São **logits** — scores brutos, **não normalizados**.

Valores maiores indicam maior preferência da rede por aquela classe, mas **não** somam 1 e **não** são probabilidades até aplicar softmax (ou deixar a loss fazer isso internamente).

---

### 7. Why do we not need to apply softmax before `nn.CrossEntropyLoss()` in PyTorch?

`nn.CrossEntropyLoss()` já combina **log-softmax + negative log-likelihood** de forma numericamente estável.

Aplicar `softmax` antes e passar probabilidades deixa a loss **incorreta** e menos estável.

**Regra:** passe **logits** direto para `CrossEntropyLoss`.

---

### 8. How do we usually convert logits into a predicted class?

Pegamos o índice da maior logit ao longo das classes:

```python
predicted = logits.argmax(dim=1)   # shape: (batch_size,)
```

Equivalente: `torch.max(logits, dim=1).indices`.

---

### 9. Why do we call `model.eval()` before validation or testing?

`model.eval()` coloca camadas como **Dropout** e **BatchNorm** em modo de inferência:

- **Dropout** desligado (não zera neurônios aleatoriamente)
- **BatchNorm** usa estatísticas do treino, não só o batch atual

A avaliação fica **estável e reproduzível**. Use com `torch.no_grad()` para não calcular gradientes.

---

### 10. Why is a custom `nn.Module` useful?

Você define **camadas no `__init__`** e **lógica no `forward`** num único objeto reutilizável.

Isso permite ramificações, gates, skip connections e integração com `.to(device)`, `train()` / `eval()`, e `model.parameters()` no otimizador.

---

### 11. What is one reason the complex model is harder to express with `nn.Sequential`?

`nn.Sequential` só encadeia camadas **em linha** (saída de uma → entrada da próxima).

No `GatedSkipNet` precisamos de:

- duas branches a partir do mesmo `h`
- mistura com gate: `gate * a + (1 - gate) * b`
- skip connection somada ao resultado
- `torch.cat` antes da camada final

Isso são **múltiplos caminhos** — não cabe numa cadeia linear simples.

---

### 12. What does `torch.cat(...)` do in the complex model?

```python
features = torch.cat([bottleneck, h], dim=1)
```

**Concatena** tensores na dimensão das features (`dim=1`):

| Tensor | Shape |
|--------|-------|
| `bottleneck` | `(batch, 32)` |
| `h` (skip) | `(batch, 64)` |
| `features` | `(batch, 96)` → entrada de `self.output` |

Em vez de somar, o modelo **junta** as duas representações lado a lado para a classificação final.


---
## Resumo — o que aprendemos

| Conceito | Em uma frase |
|----------|-------------|
| Tensor | Estrutura de dados numérica; base de tudo no PyTorch |
| Autograd | PyTorch calcula gradientes automaticamente com `.backward()` |
| Loss | Número que mede o erro; queremos minimizá-la |
| DataLoader | Entrega dados em batches para o loop de treino |
| `evaluate()` | Mede desempenho em validação sem atualizar pesos |
| Early stopping | Para o treino quando a validação para de melhorar |

### Próximos passos
- Comparar regularização (weight decay, dropout)
- Avaliar no conjunto de teste
